# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and process the FAIR\textsuperscript{2} dataset using the `mlcroissant` library, referencing all data entities by their `@id`.

### Dataset Source
The dataset source is described by a Croissant schema:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print some basic info (Note: metadata is a Python object)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}\n")
print(f"Description: {metadata.description}\n")
print("Keywords:", getattr(metadata, 'keywords', None))


## 2. Data Overview

List all available record sets and their fields using their `@id`.

In [ ]:
# List all available record sets by their @id and their fields
if hasattr(metadata, 'record_set') and metadata.record_set:
    record_sets = metadata.record_set
else:
    # Fallback (if not populated in metadata): use dataset.record_sets method
    record_sets = list(dataset.record_sets())

record_set_ids = []
for rs in dataset.record_sets():
    print(f"\nRecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):  # Single field case
            fields = [fields]
        print('  Fields:')
        for fld in fields:
            print(f"    - {fld['@id']}")
    else:
        print("  No field definitions present.")

print(f"\nAll discovered RecordSet @ids: {record_set_ids}")


## 3. Data Extraction

Load all data from the discovered record sets, referencing each by `@id`.
We construct a dictionary of DataFrames, with keys as record set `@id`s.

In [ ]:
# Extract records from each record set by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print("  No records found.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Fields in DataFrame: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field from the first populated record set and perform filtering, normalization, and grouping. Please adjust the code as needed to target specific record sets or fields of interest.

In [ ]:
# EDA: Find a numeric-like field in the first non-empty record set
# Inspect the first DataFrame populated
example_recset_id = None
example_df = None
for rsid, df in dataframes.items():
    if not df.empty:
        example_recset_id = rsid
        example_df = df
        break

if example_df is not None:
    print(f"Using RecordSet @id: {example_recset_id}")
    # Try to find any numeric column automatically (int or float)
    numeric_cols = example_df.select_dtypes(include=['int', 'float']).columns.tolist()
    if not numeric_cols:
        # Try to coerce columns to numeric (in case types were lost in loading)
        for col in example_df.columns:
            coerced = pd.to_numeric(example_df[col], errors='coerce')
            if coerced.notnull().any():
                numeric_cols.append(col)
        if numeric_cols:
            # Update column with coercion
            example_df[numeric_cols] = example_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

    if numeric_cols:
        # We'll use the first discovered numeric column
        numeric_field_id = numeric_cols[0]
        print(f"Numeric field guessed: {numeric_field_id}")
        # Filter on value > mean if possible
        threshold = example_df[numeric_field_id].mean()
        filtered_df = example_df[example_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        colnorm = f"{numeric_field_id}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, colnorm]].head())
        # Try grouping by a likely categorical field
        # Heuristic: categorical if not numeric and <20 uniques
        nonnum_cols = [c for c in example_df.columns if c != numeric_field_id]
        group_field = None
        for c in nonnum_cols:
            nunique = example_df[c].nunique(dropna=True)
            if 1 < nunique < 20:  # skip id fields with all unique
                group_field = c
                break
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} grouped by {group_field}:")
            print(grouped)
        else:
            print("No suitable categorical group field found.")
    else:
        print("No numeric fields found in sample DataFrame.")
else:
    print("No DataFrame with data found in dataframes.")


## 5. Visualization

Visualize the distribution of the filtered numeric field from above, and if grouped, plot mean by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_df is not None and numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(example_df[numeric_field_id].dropna(), bins=10, color="teal", kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we found a grouping field, plot group means
    if 'group_field' in locals() and group_field:
        group_means = example_df.groupby(group_field)[numeric_field_id].mean().sort_values()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_means.index.astype(str), y=group_means.values, palette="Set2")
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("Visualization skipped: No numeric field found.")


## 6. Conclusion

In this notebook, we loaded the FAIR\textsuperscript{2} dataset using `mlcroissant`, inspected record sets and fields via their `@id`s, extracted data, performed exploratory filtering and normalization on numeric fields, and visualized attribute distributions.
- This approach demonstrates principled FAIR access and analysis of tabular clinical datasets using the Croissant metadata model and the `mlcroissant` library.

**You can further extend this analysis by diving into other record sets or fields by their `@id`, or by implementing advanced statistical or machine learning workflows on the extracted DataFrames.**